In [48]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import os
from pathlib import Path

# ==========================================
# 1. DATA LOADING & EXTRACTION
# ==========================================
# Load the raw dataset
current_dir = Path(os.getcwd())
file_path = current_dir.parent.parent / "data" / "customer_churn_classification.xlsx"
df = pd.read_excel(file_path)

# Define y and drop target / ID variables from X
y = df['churned']
X = df.drop(columns=['churned', 'customer_id'])

# Convert any categorical features to numeric dummy codes if present
X = pd.get_dummies(X, drop_first=True)

# =====================================================================
# STEP 2: Stratified 80/20 Train and Test Split
# =====================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print("\n" + "="*50)
print("STEP 2: STRATIFICATION DISTRIBUTION ANALYSIS")
print("="*50)
print(f"Training Split Total Samples: {len(y_train)} | Canceled Customers (y=1): {sum(y_train)}")
print(f"Test Split Total Samples:     {len(y_test)}  | Canceled Customers (y=1): {sum(y_test)}")

# =====================================================================
# STEPS 3 & 4: Model Pipelines Definition with Scenarios-Specific Scalers
# =====================================================================
# StandardScaler pipelines guarantee that data fitting occurs strictly using training sets.
models = {
    'KNN': Pipeline([
        ('scaler', StandardScaler()), 
        ('knn', KNeighborsClassifier(n_neighbors=5))
    ]),
    'Gaussian Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(max_depth=3, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()), 
        ('logreg', LogisticRegression(random_state=42))
    ])
}

# =====================================================================
# STEP 5: Separately Train and Evaluate Each Model
# =====================================================================
metrics_history = {}

print("\n" + "="*50)
print("STEP 5: INDIVIDUAL MODEL PERFORMANCE TESTS")
print("="*50)

for name, model in models.items():
    # Fit the pipeline or standalone estimator
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Generate metric performance parameters
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    
    metrics_history[name] = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1}
    
    print(f"\n[{name}]")
    print(f"  Test Accuracy : {acc:.4f}")
    print(f"  Precision     : {prec:.4f} (Measures False Alarm Minimization)")
    print(f"  Recall        : {rec:.4f} (Measures Churn Capture Rate)")
    print(f"  F1-Score      : {f1:.4f}")
    print("  Confusion Matrix:")
    print(f"    [[TN: {cm[0][0]}  FP: {cm[0][1]}]\n     [FN: {cm[1][0]}  TP: {cm[1][1]}]]")

# =====================================================================
# STEP 6: Analytical Strategic Comparisons
# =====================================================================
performance_df = pd.DataFrame(metrics_history).T

highest_recall_model = performance_df['Recall'].idxmax()
highest_precision_model = performance_df['Precision'].idxmax()

print("\n" + "="*50)
print("STEP 6: COMPARATIVE ANALYSIS REPORT SUMMARY")
print("="*50)
print(f"1. Model that catches the most cancellations (Max Recall):     {highest_recall_model} ({performance_df.loc[highest_recall_model, 'Recall']:.4f})")
print(f"2. Model that minimizes false alerts (Max Precision):        {highest_precision_model} ({performance_df.loc[highest_precision_model, 'Precision']:.4f})")
print("3. Would you select the same model for both objectives?")
print("   -> NO. Choosing a single model is blocked by the Precision-Recall trade-off curve.")
print("      Optimizing for high recall increases false alerts, while maximizing precision creates strict misses.")

# =====================================================================
# STEP 7: Inspect Model Internals (What Was Learned)
# =====================================================================
print("\n" + "="*50)
print("STEP 7: INTERPRETING LEARNED PARAMETERS")
print("="*50)

# KNN Parameter Display
print(f"\n[KNN Parameters]")
print(f"  - Metric Rule Used: {models['KNN'].named_steps['knn'].metric.upper()}")
print(f"  - Configured Nearest Neighbors (K): {models['KNN'].named_steps['knn'].n_neighbors}")

# Decision Tree Splits Exploration
print("\n[Decision Tree Layout Structural Map]")
tree_rules = export_text(models['Decision Tree'], feature_names=list(X.columns))
print(tree_rules)

# Random Forest Ensemble Node Importances
print("\n[Random Forest Top Feature Weight Coefficients]")
rf_importances = pd.Series(
    models['Random Forest'].feature_importances_, index=X.columns
).sort_values(ascending=False)
for rank, (feat, val) in enumerate(rf_importances.items(), 1):
    print(f"  Rank {rank}: Feature '{feat}' holds an importance metric of {val:.4f}")

# Logistic Regression Vector Coordinates weights
print("\n[Logistic Regression Coefficients - Log-Odds Scales]")
lr_coefs = pd.Series(
    models['Logistic Regression'].named_steps['logreg'].coef_[0], index=X.columns
).sort_values(ascending=False)
for feat, val in lr_coefs.items():
    print(f"  Feature '{feat}': Coefficient = {val:.4f} (Higher positive indicates direct risk increase)")

# =====================================================================
# STEP 8: Strategic Recommendation Summary Brief
# =====================================================================
print("\n" + "="*50)
print("STEP 8: RECOVERY RETENTION CAMPAIGN RECOMMENDATION")
print("="*50)
print(f"For this campaign, you should select the model with the best recall: {highest_recall_model}.")
print("Reasoning: In retention campaigns, missing an at-risk customer (a False Negative) is far more costly ")
print("than giving a small promotional discount to a customer who wasn't actually going to leave (a False Positive).")
print("Because the costs of customer acquisition dramatically outpace small retention incentive payouts, ")
print("the model offering the highest sensitivity (Recall) makes the best financial anchor for targeting accounts.")
print("="*50 + "\n")


STEP 2: STRATIFICATION DISTRIBUTION ANALYSIS
Training Split Total Samples: 400 | Canceled Customers (y=1): 112
Test Split Total Samples:     100  | Canceled Customers (y=1): 28

STEP 5: INDIVIDUAL MODEL PERFORMANCE TESTS

[KNN]
  Test Accuracy : 0.7500
  Precision     : 0.5652 (Measures False Alarm Minimization)
  Recall        : 0.4643 (Measures Churn Capture Rate)
  F1-Score      : 0.5098
  Confusion Matrix:
    [[TN: 62  FP: 10]
     [FN: 15  TP: 13]]

[Gaussian Naive Bayes]
  Test Accuracy : 0.7800
  Precision     : 0.6000 (Measures False Alarm Minimization)
  Recall        : 0.6429 (Measures Churn Capture Rate)
  F1-Score      : 0.6207
  Confusion Matrix:
    [[TN: 60  FP: 12]
     [FN: 10  TP: 18]]

[Decision Tree]
  Test Accuracy : 0.7000
  Precision     : 0.4667 (Measures False Alarm Minimization)
  Recall        : 0.5000 (Measures Churn Capture Rate)
  F1-Score      : 0.4828
  Confusion Matrix:
    [[TN: 56  FP: 16]
     [FN: 14  TP: 14]]

[Random Forest]
  Test Accuracy : 0.